<a href="https://colab.research.google.com/github/atulyapanda70/ph1050-group1/blob/main/part1_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Real NASA Planetary Fact Sheet Data for the 8 Planets
planets_data = {
    "name":  np.array(["Mercury", "Venus", "Earth", "Mars", "Jupiter", "Saturn", "Uranus", "Neptune"]),
    "a_AU":  np.array([0.387098, 0.723332, 1.000000, 1.523679, 5.204400, 9.582600, 19.21840, 30.11000]),
    "e":     np.array([0.205630, 0.006772, 0.016708, 0.093400, 0.048900, 0.056500, 0.046381, 0.009456]),
    "T_obs": np.array([0.240846, 0.615198, 1.000017, 1.880849, 11.86200, 29.45710, 84.01200, 164.7885])
}

GM = 4.0 * np.pi**2

def derivs(state):
    (x, y, vx, vy) = state
    r = (x**2 + y**2)**0.5
    return np.array([vx, vy, -GM * x / r**3, -GM * y / r**3])

def rk4_step(state, dt):
    k1 = derivs(state)
    k2 = derivs(state + 0.5 * dt * k1)
    k3 = derivs(state + 0.5 * dt * k2)
    k4 = derivs(state + dt * k3)
    return state + (dt / 6.0) * (k1 + 2.0*k2 + 2.0*k3 + k4)

names = planets_data["name"]
a_arr = planets_data["a_AU"]
e_arr = planets_data["e"]
T_obs = planets_data["T_obs"]

# Vectorized calculation of aphelion distance r = a(1+e) and speed v = sqrt(GM(2/r - 1/a))
r_ap = a_arr * (1.0 + e_arr)
v_ap = np.sqrt(GM * (2.0 / r_ap - 1.0 / a_arr))
T_sim = np.zeros(len(names))

for i in range(len(names)):
    state = np.array([r_ap[i], 0.0, 0.0, v_ap[i]])
    T_est = a_arr[i]**1.5 # T= 2̀̄̄(r^3/GM)^0.5, GM=4̀̄̄^2 so T=r^1.5
    dt_p  = T_est / 4000.0
    t = 0.0
    while True:
        prev = state.copy()
        state = rk4_step(state, dt_p)
        t += dt_p
        if t > 0.5 * T_est and prev[1] < 0.0 and state[1] >= 0.0:
            frac = -prev[1] / (state[1] - prev[1])
            T_sim[i] = (t - dt_p) + frac * dt_p
            break

rel_err = np.abs(T_sim - T_obs) / T_obs * 100.0

print(f"{'Planet':<10} {'a (AU)':<10} {'e':<10} {'T_sim (yr)':<13} {'T_obs (yr)':<13} {'Rel Error (%)'}")
print("-" * 72)
for i in range(len(names)):
    print(f"{names[i]:<10} {a_arr[i]:<10.6f} {e_arr[i]:<10.6f} {T_sim[i]:<13.6f} {T_obs[i]:<13.6f} {rel_err[i]:.4f}%")

max_i = np.argmax(rel_err)
print(f"\nLargest Relative Error: {names[max_i]} ({rel_err[max_i]:.4f}%)")

Planet     a (AU)     e          T_sim (yr)    T_obs (yr)    Rel Error (%)
------------------------------------------------------------------------
Mercury    0.387098   0.205630   0.240842      0.240846      0.0019%
Venus      0.723332   0.006772   0.615186      0.615198      0.0019%
Earth      1.000000   0.016708   1.000000      1.000017      0.0017%
Mars       1.523679   0.093400   1.880790      1.880849      0.0032%
Jupiter    5.204400   0.048900   11.872878     11.862000     0.0917%
Saturn     9.582600   0.056500   29.663681     29.457100     0.7013%
Uranus     19.218400  0.046381   84.251151     84.012000     0.2847%
Neptune    30.110000  0.009456   165.221337    164.788500    0.2627%

Largest Relative Error: Saturn (0.7013%)
